In [1]:
import pandas as pd
import random

templates = {
    'vendas': {
        's': ['', 'Olá', 'Bom dia', 'Gostaria de saber', 'Por favor'],
        'a': ['quero comprar', 'qual o preco do', 'tem cupom para', 'como faco para adquirir', 'desejo orcamento de'],
        'o': ['sofa retratil 3 lugares', 'conjunto de mesa de jantar', 'guarda roupa casal', 'painel para tv', 'colchao queen size']
    },
    'suporte': {
        's': ['', 'Oi', 'Preciso de ajuda', 'Por gentileza', 'Socorro'],
        'a': ['como montar o', 'onde baixo o manual do', 'estou com duvida no', 'veio faltando parafuso no', 'preciso de assistencia para'],
        'o': ['armario de cozinha', 'rack da sala', 'berco do bebe', 'esquema de montagem', 'manual da estante']
    },
    'trocas_devolucoes': {
        's': ['', 'Olá', 'Por favor', 'Gostaria de solicitar', 'Quero abrir'],
        'a': ['preciso trocar o', 'quero devolver a', 'como solicito o estorno do', 'desejo solicitar a troca da', 'como funciona a devolucao do'],
        'o': ['produto com defeito', 'mesa que veio arranhada', 'cadeira no prazo de 7 dias', 'pedido cancelado', 'item com avaria']
    },
    'reclamacoes': {
        's': ['', 'Urgente', 'Pessimo atendimento', 'Absurdo', 'Quero registrar'],
        'a': ['estou indignado com o', 'quero fazer uma queixa do', 'estou reclamando do', 'produto veio quebrado e o', 'atendimento horrivel do'],
        'o': ['atraso na minha entrega', 'servico de montagem', 'sac que nao responde', 'pos venda da loja', 'estado do meu movel']
    },
    'logistica_entregas': {
        's': ['', 'Olá', 'Bom dia', 'Por gentileza', 'Preciso saber'],
        'a': ['onde esta o meu', 'qual o prazo de entrega do', 'como rastreio a', 'qual a transportadora do', 'quando chega o'],
        'o': ['meu pedido', 'codigo de rastreamento', 'movel comprado', 'status do envio', 'agendamento da entrega']
    }
}

amostras = []
random.seed(42)

for intencao, comp in templates.items():
    for _ in range(20):  # Total: 100 amostras (20 por classe)
        s = random.choice(comp['s'])
        a = random.choice(comp['a'])
        o = random.choice(comp['o'])
        frase = f"{s} {a} {o}".strip().capitalize()
        amostras.append({'texto': frase, 'intencao': intencao})

df_moveis = pd.DataFrame(amostras)
df_moveis.to_csv('dataset_moveis_100.csv', index=False, encoding='utf-8')

print(" Dataset 'dataset_moveis_100.csv' criado com 100 frases distribuidas em 5 intencoes!")


 Dataset 'dataset_moveis_100.csv' criado com 100 frases distribuidas em 5 intencoes!


In [26]:
qimport numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

# 1. Carregar dataset do CSV
df = pd.read_csv('dataset_moveis_100.csv')

# 2. Divisão Treino e Teste
X_train, X_test, y_train, y_test = train_test_split(
    df['texto'], df['intencao'], test_size=0.30, random_state=42, stratify=df['intencao']
)

# TODO 1: Monte a Pipeline utilizando TfidfVectorizer e KNeighborsClassifier(n_neighbors=3, metric='cosine')
pipeline_knn = Pipeline([
    ('vectorizer', TfidfVectorizer(ngram_range=(1,2))),
    ('classifier', KNeighborsClassifier(n_neighbors=3, metric='cosine'))
])

# TODO 2: Treine a pipeline com os dados de treino (X_train, y_train)
pipeline_knn.fit(X_train, y_train)

# TODO 3: Gere as predicoes nos dados de teste e exiba o classification_report e a confusion_matrix
y_pred = pipeline_knn.predict(X_test)
print("\n--- Classification Report (KNN) ---")
print(classification_report(y_test, y_pred))
print("\n--- Confusion Matrix (KNN) ---")
print(confusion_matrix(y_test, y_pred))

LIMIAR_CONFIANCA = 0.50

print("\n=== INICIANDO BATERIA DE TESTES (10 INPUTS OBRIGATÓRIOS) ===")

for i in range(1, 11):
    print(f"\n[Teste {i}/10]")

    # TODO 4: Solicite a frase do usuario via teclado
    frase = input("Digite a frase do cliente: ").strip()

    # TODO 5: Extraia as probabilidades e a classe prevista usando predict_proba e predict
    probs = pipeline_knn.predict_proba([frase])[0]
    maior_prob = np.max(probs)
    intencao = pipeline_knn.predict([frase])[0]

    # TODO 6: Aplique a regra de decisao:
    # Se maior_prob >= LIMIAR_CONFIANCA: imprima a intencao e a probabilidade.
    # Senao: imprima o Fallback encaminhando para atendimento humano.

    if maior_prob >= LIMIAR_CONFIANCA:
        print(f"Intenção prevista: {intencao} (Probabilidade: {maior_prob:.2f})")
    else:
        print("Desculpe, não entendi sua solicitação. Encaminhando você para um atendente humano...")


--- Classification Report (KNN) ---
                    precision    recall  f1-score   support

logistica_entregas       1.00      1.00      1.00         6
       reclamacoes       1.00      1.00      1.00         6
           suporte       1.00      1.00      1.00         6
 trocas_devolucoes       1.00      1.00      1.00         6
            vendas       1.00      1.00      1.00         6

          accuracy                           1.00        30
         macro avg       1.00      1.00      1.00        30
      weighted avg       1.00      1.00      1.00        30


--- Confusion Matrix (KNN) ---
[[6 0 0 0 0]
 [0 6 0 0 0]
 [0 0 6 0 0]
 [0 0 0 6 0]
 [0 0 0 0 6]]

=== INICIANDO BATERIA DE TESTES (10 INPUTS OBRIGATÓRIOS) ===

[Teste 1/10]
Digite a frase do cliente: ajuda
Intenção prevista: suporte (Probabilidade: 0.67)

[Teste 2/10]
Digite a frase do cliente: devolucao
Intenção prevista: trocas_devolucoes (Probabilidade: 1.00)

[Teste 3/10]
Digite a frase do cliente: comprar
Inten

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

# 1. Carregar dataset do CSV
df = pd.read_csv('dataset_moveis_100.csv')

# 2. Divisão Treino e Teste
X_train, X_test, y_train, y_test = train_test_split(
    df['texto'], df['intencao'], test_size=0.30, random_state=42, stratify=df['intencao']
)


from sklearn.tree import DecisionTreeClassifier

pipeline_arvore = Pipeline([
    ('vectorizer', TfidfVectorizer(ngram_range=(1,2))),
    ('classifier', DecisionTreeClassifier(random_state=42))
])


pipeline_arvore.fit(X_train,y_train)
pred = pipeline_arvore.predict(X_test)

print("\n--- Classification Report (Decision Tree) ---")
print(classification_report(y_test, pred))
print("\n--- Confusion Matrix (Decision Tree) ---")
print(confusion_matrix(y_test, pred))

LIMIAR_CONFIANCA = 0.50

print("\n=== INICIANDO BATERIA DE TESTES (8 INPUTS OBRIGATÓRIOS - ÁRVORE DE DECISÃO) ===")

for i in range(1, 9): # 8 inputs as requested
    print(f"\n[Teste {i}/8]")

    frase = input("Digite a frase do cliente: ").strip()

    probs = pipeline_arvore.predict_proba([frase])[0]
    maior_prob = np.max(probs)
    intencao = pipeline_arvore.predict([frase])[0]

    if maior_prob >= LIMIAR_CONFIANCA:
        print(f"Intenção prevista: {intencao} (Probabilidade: {maior_prob:.2f})")
    else:
        print("Desculpe, não entendi sua solicitação. Encaminhando você para um atendente humano...")

# Relatório de Avaliação NLU - SAC Móveis Residenciais

## 1. Tabela Comparativa de Métricas (Dados de Teste)

| Modelo | Acurácia Geral | F1-Score (Weighted) | Principais Erros na Matriz |
| :--- | :--- | :--- | :--- |
| **KNN (K=3)** | 1.00 % | 1.00 % | Nenhuma, todas as classes foram perfeitamente classificadas. |
| **Decision Tree** | 1.00 % | 1.00 % | Nenhuma, todas as classes foram perfeitamente classificadas. |

## 2. Análise dos Testes de Entrada (`input()`)

- **Comportamento do KNN (10 testes):**
    O modelo KNN demonstrou uma excelente capacidade de generalização para as frases inseridas. Frases isoladas ou com pequenas variações como "ajuda", "devolucao", "comprar" e "quero ajuda" foram classificadas corretamente para suas respectivas intenções ('suporte', 'trocas_devolucoes', 'vendas'). As probabilidades de confiança variaram entre 0.67 e 1.00, mostrando um nível adequado de certeza. O fallback (mensagem para atendente humano) foi acionado corretamente para entradas sem sentido ("jfjfa]fsw", "mniog jioçbsd"), indicando robustez para lidar com ruídos ou entradas fora do domínio.

- **Comportamento da Decision Tree (8 testes):**
    A Árvore de Decisão, apesar de ter métricas perfeitas no conjunto de teste, apresentou um comportamento menos consistente nos testes interativos. A palavra "ajuda" foi sistematicamente classificada como 'trocas_devolucoes' (com 1.00 de probabilidade), mesmo sendo associada a 'suporte' no conjunto de treinamento. Frases como "comprar" também foram erroneamente classificadas como 'trocas_devolucoes'. Uma entrada vazia resultou em uma classificação 'trocas_devolucoes' com 100% de probabilidade, o que seria um erro grave em um sistema real (o esperado seria um fallback). O fallback não foi acionado para a maioria das entradas incorretas ou ambíguas, pois o modelo atribuiu 100% de probabilidade a uma classe errada, sugerindo uma tendência a 'super-confiança' em classificações incorretas para entradas fora do contexto exato de treino.

## 3. Veredito Final

- **Melhor modelo para este projeto:** KNN (K=3)
- **Justificativa técnica:**
    Embora ambos os modelos tenham alcançado 100% de acurácia e F1-Score ponderado no conjunto de dados de teste (indicando que são capazes de aprender perfeitamente o dataset sintético), a performance nos testes interativos com `input()` revelou diferenças cruciais. O KNN demonstrou maior robustez e capacidade de generalização para frases que não estavam idênticas ou com contexto completo presente no treinamento. Ele classificou corretamente as intenções para palavras-chave isoladas e ativou o fallback de maneira apropriada para entradas sem sentido. A Árvore de Decisão, por outro lado, mostrou-se excessivamente específica aos padrões aprendidos, resultando em classificações incorretas para frases simples (como "ajuda" e "comprar") e não acionando o fallback quando necessário. Portanto, o KNN é o modelo mais indicado devido à sua melhor adaptabilidade e desempenho em um cenário de uso real com entradas mais variadas.